# Getting documents

Goal: turn a GitHub docs repo into a plain list of Python dicts we can search later.
First by hand, then with a library that does the same thing in three lines.

## Download the repo as a zip

GitHub hands out any branch as a zip, so one HTTP request gets us the whole repo.
No `git clone`, no local checkout to keep in sync.

In [ ]:
!uv add requests

In [ ]:
import requests

In [ ]:
repo_owner = "evidentlyai"
repo_name = "docs"
repo_branch = "main"

zip_url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{repo_branch}.zip"
zip_response = requests.get(zip_url, allow_redirects=True)

Should be a few hundred KB. If it comes back tiny we probably got an error page, not a zip.

In [ ]:
len(zip_response.content)

## Open the zip without unpacking it

`BytesIO` lets `zipfile` read straight from the bytes we just downloaded, so nothing touches disk.

In [ ]:
import io
import zipfile

zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))

In [ ]:
zip_archive

In [ ]:
zip_archive.namelist()

## Pull out one file first

Much easier to get a single file right than to debug a loop over ninety of them.

In [ ]:
file_name = "docs-main/docs/platform/alerts.mdx"
mdx_file = zip_archive.open(file_name)

In [ ]:
mdx_content = mdx_file.read().decode("utf8")

## Split the frontmatter from the text

Every `.mdx` page starts with a small YAML header (title, description). `frontmatter` peels that
off, so we get the metadata as a dict and the body as clean text instead of the header bleeding
into the content we want to search.

In [ ]:
!uv add python-frontmatter

In [ ]:
import frontmatter

post = frontmatter.loads(mdx_content)
print(post.content[:500])

In [ ]:
post.metadata

In [ ]:
post.content

## Clean up the file path

The zip nests everything under `docs-main/`, which isn't part of the real path in the repo.
Split once and keep the tail.

In [ ]:
file_name.split("/", maxsplit=1)

In [ ]:
filename_corrected = file_name.split("/", maxsplit=1)[-1]

In [ ]:
filename_corrected

## Shape one document

This dict is the format we want everything in: the text to search over, plus the metadata we'd
want to show alongside a result.

In [ ]:
doc = {
    "content": post.content,
    "title": post.metadata.get("title"),
    "description": post.metadata.get("description"),
    "file_name": filename_corrected,
}

In [ ]:
doc

## Run it over the whole repo

Same steps as above, now in a loop. Anything that isn't `.md` or `.mdx` gets skipped - images and
config files are just noise for search.

In [ ]:
documents = []

with zipfile.ZipFile(io.BytesIO(zip_response.content)) as zip_ref:
    for file_path in zip_ref.namelist():
        if not file_path.endswith((".md", ".mdx")):
            continue

        with zip_ref.open(file_path) as file:
            content = file.read().decode("utf-8")
            post = frontmatter.loads(content)

            doc = {
                "content": post.content,
                "title": post.metadata.get("title"),
                "description": post.metadata.get("description"),
                "filename": file_path.split("/", 1)[-1],
            }
            documents.append(doc)

In [ ]:
len(documents)

## Same job, done by a library

`gitsource` wraps everything above: download, unzip, filter by extension, parse the frontmatter.
Worth doing it by hand once to know what it's doing for you, then use this.

In [ ]:
!uv add gitsource

In [ ]:
%pip install gitsource

In [ ]:
%pip show gitsource

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

In [ ]:
files[0]

In [ ]:
files[10].content

In [ ]:
files[10].parse

`.parse()` gives back the same dict shape we built manually, so this one line replaces the whole
loop. Note it reuses the `documents` name, so the two cells below now read this version, not the
hand-built one.

In [ ]:
documents = [f.parse() for f in files]

In [ ]:
len(documents)

In [ ]:
documents[10]